In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr


In [ ]:
def plot_interactive_reconstructions(ds, epoches=None, variables=None, samples=None):
    # Resolve selections
    variables = ds.variable.values if variables is None else ds.variable.values[variables]
    epochs    = ds.epoch.values    if epoches  is None else ds.epoch.values[epoches]
    samples   = ds.sample.values   if samples  is None else np.asarray(samples)

    # Convert to plain Python scalars once — avoids numpy hashability issues everywhere
    var_list    = variables.tolist()
    epoch_list  = epochs.tolist()
    sample_list = samples.tolist()

    # ── 1. Timestamps — only for requested samples ────────────────────────────
    sample_dates = {}
    ts_block = ds.timestamp.sel(sample=sample_list).compute()
    for sample in sample_list:
        ts = ts_block.sel(sample=sample).values.squeeze().item()
        sample_dates[sample] = pd.Timestamp(ts, unit='s').strftime('%Y-%m-%d %H:%M')

    # ── 2. Slice FIRST, then compute — only load what we need ─────────────────
    print("Loading data subset…")
    ds_sub = ds.sel(epoch=epoch_list, sample=sample_list, variable=var_list)

    orig_all  = ds_sub.original.squeeze(["time", "ensemble"]).compute()   # small now
    recon_all = ds_sub.reconstruction.squeeze(["time", "ensemble"]).compute()

    # Convert to NumPy once — shape: (epoch, sample, variable, y, x)
    orig_np  = orig_all.values.astype(np.float32)
    recon_np = recon_all.values.astype(np.float32)

    # Integer index maps into the *subset* arrays
    var_idx   = {v: i for i, v in enumerate(var_list)}
    samp_idx  = {s: i for i, s in enumerate(sample_list)}
    epoch_idx = {e: i for i, e in enumerate(epoch_list)}

    # ── 3. Colour scales — computed from the subset only ─────────────────────
    print("Computing colour scales…")
    var_scales  = {}
    diff_scales = {}
    for var in var_list:
        vi = var_idx[var]
        o  = orig_np [:, :, vi]   # (epoch, sample, y, x)
        r  = recon_np[:, :, vi]
        var_scales[var]  = (float(min(o.min(), r.min())), float(max(o.max(), r.max())))
        diff_scales[var] = (-float(np.abs(o - r).max()),  float(np.abs(o - r).max()))

    # ── 4. Plot loop — pure NumPy, zero Dask/xarray calls ────────────────────
    print("Plotting…")
    for epoch in epoch_list:
        ei = epoch_idx[epoch]

        for var in var_list:
            vi         = var_idx[var]
            vmin, vmax = var_scales[var]
            dmin, dmax = diff_scales[var]
            cmap_main  = 'Blues' if var == 'tp' else 'magma'
            n          = len(sample_list)

            fig, axes = plt.subplots(n, 3, figsize=(15, 4 * n), constrained_layout=True)
            if n == 1:
                axes = np.expand_dims(axes, axis=0)

            im_main = im_diff = None
            for s_idx, sample in enumerate(sample_list):
                si    = samp_idx[sample]
                orig  = np.flipud(orig_np [ei, si, vi].astype(np.float64))
                recon = np.flipud(recon_np[ei, si, vi].astype(np.float64))
                diff  = orig - recon
                rmse  = float(np.sqrt(np.mean(diff ** 2)))

                diff_min = float(diff.min())
                diff_max = float(diff.max())
                min_loc  = np.unravel_index(np.argmin(diff), diff.shape)
                max_loc  = np.unravel_index(np.argmax(diff), diff.shape)
                date_str = sample_dates[sample]

                # Original
                ax = axes[s_idx, 0]
                im_main = ax.imshow(orig, cmap=cmap_main, vmin=vmin, vmax=vmax)
                ax.set_title(f"Epoch {epoch} | {var} | {date_str}\n(Original)")
                ax.axis('off')

                # Reconstruction
                ax = axes[s_idx, 1]
                ax.imshow(recon, cmap=cmap_main, vmin=vmin, vmax=vmax)
                ax.set_title(f"Epoch {epoch} | {var} | {date_str}\n(Reconstructed)")
                ax.axis('off')

                # Difference
                ax = axes[s_idx, 2]
                im_diff = ax.imshow(diff, cmap='RdBu_r', vmin=dmin, vmax=dmax)
                ax.set_title(f"Epoch {epoch} | {var} | {date_str}\n(Difference  |  RMSE: {rmse:.4f})")
                ax.axis('off')

                for loc, marker, color, label, offset in [
                    (min_loc, 'v', 'blue', f'Min: {diff_min:.4f}', (6,   6)),
                    (max_loc, '^', 'red',  f'Max: {diff_max:.4f}', (6, -14)),
                ]:
                    ax.plot(loc[1], loc[0], marker, color=color, markersize=7,
                            markeredgecolor='white', markeredgewidth=0.8, label=label)
                    ax.annotate(
                        label.replace(': ', '\n'),
                        xy=(loc[1], loc[0]), xytext=offset, textcoords='offset points',
                        color=color, fontsize=7, fontweight='bold',
                        bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.6, ec='none'),
                    )
                ax.legend(loc='lower right', fontsize=7, framealpha=0.7, markerscale=0.9)

            fig.colorbar(im_main, ax=axes[:, :2], orientation='vertical',
                         fraction=0.02, pad=0.04).set_label(f'Value Scale for {var}')
            fig.colorbar(im_diff, ax=axes[:, 2],  orientation='vertical',
                         fraction=0.04, pad=0.04).set_label(f'Difference Scale for {var}')
            plt.show()

In [ ]:
import xarray as xr


file_paths = {
   'mse_kl_04_0_20260417_092115':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/ldm-vae-answer/mse_kl_04_0_20260417_092115/samples.zarr',
   #'02-l2-no-norm/mse_kl_04_0_20260609_024857':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/02-l2-no-norm/mse_kl_04_0_20260609_024857/samples.zarr',
   #'03-lola-no-norm/lola_dcae_20260609_024929':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/03-lola-no-norm/lola_dcae_20260609_024929/samples.zarr',
   #'04-qrl-no-norm/qrl_dcae_20260609_024937':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/04-qrl-no-norm/qrl_dcae_20260609_024937/samples.zarr',
   #'05-lola-uniform/lola_uniform_20260609_024945':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/05-lola-uniform/lola_uniform_20260609_024945/samples.zarr',
   #'06-lola-winds/lola_winds_20260609_024952':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/06-lola-winds/lola_winds_20260609_024952/samples.zarr',
   #'07-lola-32x/lola_32x_20260609_025000':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/07-lola-32x/lola_32x_20260609_025000/samples.zarr',
      #'08-lola-z64/lola_z64_20260609_025027':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/08-lola-z64/lola_z64_20260609_025027/samples.zarr',
   #'09-winds-focal/winds_focal_20260609_025034':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/09-winds-focal/winds_focal_20260609_025034/samples.zarr',
      #'09-winds-focal-z32x32x32-20260609_145310':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/09-winds-focal/09-winds-focal-z32x32x32-20260609_145310/samples.zarr',
      #'09-winds-focal-z64x16x16-20260609_145317':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/09-winds-focal/09-winds-focal-z64x16x16-20260609_145317/samples.zarr',
   #'10-arcsinh-z64/arcsinh_z64_20260609_025040':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/10-arcsinh-z64/arcsinh_z64_20260609_025040/samples.zarr',
   #'11-qrl-32x/qrl_32x_20260609_025046':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/11-qrl-32x/qrl_32x_20260609_025046/samples.zarr',
   #'12-rombach-ae/kl_f8_20260609_032527':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/12-rombach-ae/kl_f8_20260609_032527/samples.zarr',
   #'12-rombach-ae/vq_f8_20260609_032535':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/12-rombach-ae/vq_f8_20260609_032535/samples.zarr',
   '13-qrl-vae-winds-focal-z64x16x16-20260609_150346':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/13-qrl-vae-winds-focal/13-qrl-vae-winds-focal-z64x16x16-20260609_150346/samples.zarr',
   #'15-qrl-vae-winds-kl-z32x32x32-20260610_102718':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/15-qrl-vae-winds-kl/15-qrl-vae-winds-kl-z32x32x32-20260610_102718/samples.zarr',
   #'15-qrl-vae-winds-kl-z64x16x16-20260610_102719':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/15-qrl-vae-winds-kl/15-qrl-vae-winds-kl-z64x16x16-20260610_102719/samples.zarr',
   #'16-qrl-vae-winds-kl-mmd-z32x32x32-20260610_102720':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/16-qrl-vae-winds-kl-mmd/16-qrl-vae-winds-kl-mmd-z32x32x32-20260610_102720/samples.zarr',
   #'16-qrl-vae-winds-kl-mmd-z64x16x16-20260610_102721':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/16-qrl-vae-winds-kl-mmd/16-qrl-vae-winds-kl-mmd-z64x16x16-20260610_102721/samples.zarr',
   #'17-lola-vae-kl-mmd-z64-20260610_102722/':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/17-lola-vae-kl/17-lola-vae-kl-mmd-z64-20260610_102722/samples.zarr',
   #'17-lola-vae-kl-z64-20260610_102724':'/mnt/tier2/project/p200177/u101329/DE371_bis/diffusion-interpolation/_work/17-lola-vae-kl/17-lola-vae-kl-z64-20260610_102724/samples.zarr',
}



In [ ]:
for label, file_path in file_paths.items():
    print(f"\n{'='*60}")
    print(f"  {label}  —  {file_path.split('/')[-2]}")
    print(f"{'='*60}")
    try:
        ds = xr.open_zarr(file_path, zarr_format=3, consolidated=True)
        print(ds)
        plot_interactive_reconstructions(ds, epoches=[-1], variables=[0])
    except Exception as e:
        print(f"  ERROR: {e}")

In [ ]:
for label, file_path in file_paths.items():
    print(f"\n{'='*60}")
    print(f"  {label}  —  {file_path.split('/')[-2]}")
    print(f"{'='*60}")
    try:
        ds = xr.open_zarr(file_path, zarr_format=3, consolidated=True)
        print(ds)
        plot_interactive_reconstructions(ds, epoches=[-1], variables=[1])
    except Exception as e:
        print(f"  ERROR: {e}")

In [ ]:
for label, file_path in file_paths.items():
    print(f"\n{'='*60}")
    print(f"  {label}  —  {file_path.split('/')[-2]}")
    print(f"{'='*60}")
    try:
        ds = xr.open_zarr(file_path, zarr_format=3, consolidated=True)
        print(ds)
        plot_interactive_reconstructions(ds, epoches=[-1], variables=[2])
    except Exception as e:
        print(f"  ERROR: {e}")

In [ ]:
for label, file_path in file_paths.items():
    print(f"\n{'='*60}")
    print(f"  {label}  —  {file_path.split('/')[-2]}")
    print(f"{'='*60}")
    try:
        ds = xr.open_zarr(file_path, zarr_format=3, consolidated=True)
        print(ds)
        plot_interactive_reconstructions(ds, epoches=[-1], variables=[3])
    except Exception as e:
        print(f"  ERROR: {e}")

In [ ]:
for label, file_path in file_paths.items():
    print(f"\n{'='*60}")
    print(f"  {label}  —  {file_path.split('/')[-2]}")
    print(f"{'='*60}")
    try:
        ds = xr.open_zarr(file_path, zarr_format=3, consolidated=True)
        print(ds)
        plot_interactive_reconstructions(ds, epoches=[-1], variables=[4])
    except Exception as e:
        print(f"  ERROR: {e}")

In [ ]:
for label, file_path in file_paths.items():
    print(f"\n{'='*60}")
    print(f"  {label}  —  {file_path.split('/')[-2]}")
    print(f"{'='*60}")
    try:
        ds = xr.open_zarr(file_path, zarr_format=3, consolidated=True)
        print(ds)
        plot_interactive_reconstructions(ds, epoches=[-1], variables=[5])
    except Exception as e:
        print(f"  ERROR: {e}")